# Cleaner Verification — before vs. after

**Purpose:** Visually confirm `app/ingestion/cleaner.py` correctly removes the running header found during exploration (`01_ingestion_exploration.ipynb`), without damaging pages that never had it.

Recap of findings that drove `cleaner.py`:
- Repeated 2-line header ("UNIVERSITY OF EDUCATION, LAHORE" + "FALL ...") on roughly half of sampled pages
- No repeated footer pattern
- No broken line breaks, no encoding issues

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from app.ingestion.parser import extract_pdf_pages
from app.ingestion.cleaner import clean_pages

PDF_PATH = project_root / "data" / "raw_documents" / "Prospectus - FALL 2026 (29-07-2026).pdf"

pages = extract_pdf_pages(PDF_PATH)
cleaned_pages = clean_pages(pages)

print(f"Loaded and cleaned {len(cleaned_pages)} pages.")

Loaded and cleaned 382 pages.


## Before / after on a header page vs. a non-header page

In [2]:
def show_before_after(page_number: int, max_chars: int = 500) -> None:
    page = cleaned_pages[page_number - 1]
    print(f"{'#' * 70}")
    print(f"PAGE {page_number}")
    print(f"{'#' * 70}")
    print(f"--- BEFORE (raw, {page['char_count']} chars) ---")
    print(page["text"][:max_chars])
    print()
    print(f"--- AFTER (cleaned, {page['cleaned_char_count']} chars) ---")
    print(page["cleaned_text"][:max_chars])
    print()


show_before_after(13)   # had the header -> should be gone
show_before_after(53)   # no header -> should look almost identical

######################################################################
PAGE 13
######################################################################
--- BEFORE (raw, 1002 chars) ---
UNIVERSITY OF EDUCATION, LAHORE
FALL
SPORTS
The Directorate of Sports promotes physical fitness, sportsmanship, teamwork, and 
healthy lifestyles among students through organized sports activities and competitions at 
campus, inter-campus, and national levels. The Directorate provides opportunities for 
students to participate in a wide range of indoor and outdoor sports while fostering 
discipline, leadership, and competitive spirit.
Mission
Ÿ
Promote physical and mental well-being through spor

--- AFTER (cleaned, 961 chars) ---
SPORTS
The Directorate of Sports promotes physical fitness, sportsmanship, teamwork, and
healthy lifestyles among students through organized sports activities and competitions at
campus, inter-campus, and national levels. The Directorate provides opportunities for
students to par

## Document-wide sanity check

In [3]:
still_has_header = [
    p["page_number"] for p in cleaned_pages
    if "UNIVERSITY OF EDUCATION, LAHORE" in p["cleaned_text"]
]

total_raw_chars = sum(p["char_count"] for p in cleaned_pages)
total_cleaned_chars = sum(p["cleaned_char_count"] for p in cleaned_pages)
removed_chars = total_raw_chars - total_cleaned_chars

print(f"Pages still containing header after cleaning: {len(still_has_header)} (expect 0)")
print(f"Total raw chars: {total_raw_chars}")
print(f"Total cleaned chars: {total_cleaned_chars}")
print(f"Chars removed: {removed_chars} ({removed_chars / total_raw_chars:.1%} of document)")

Pages still containing header after cleaning: 302 (expect 0)
Total raw chars: 622275
Total cleaned chars: 606853
Chars removed: 15422 (2.5% of document)


## DIAGNOSTIC — why is the phrase still present on 302 pages?

The `strip_running_header` function only removes the phrase when it's the
very FIRST two lines of a page. If 302 pages still contain the phrase
after cleaning, that most likely means the phrase also appears elsewhere
in the BODY text (e.g. affiliated colleges list, degree titles,
letterhead repeated per program) - not just as a running header.

This cell shows exactly WHERE and HOW MANY TIMES the phrase occurs on a
few of the still-flagged pages, with surrounding context, so we can see
the real pattern instead of guessing.

In [4]:
import re

PHRASE = "UNIVERSITY OF EDUCATION, LAHORE"

# Count occurrences per flagged page, sorted by how many times it appears
occurrence_counts = []
for p in cleaned_pages:
    count = p["cleaned_text"].count(PHRASE)
    if count > 0:
        occurrence_counts.append((p["page_number"], count))

occurrence_counts.sort(key=lambda x: -x[1])

print(f"Total flagged pages: {len(occurrence_counts)}")
print(f"Top 10 pages by occurrence count:")
for page_num, count in occurrence_counts[:10]:
    print(f"  Page {page_num}: {count} occurrences")

avg_occurrences = sum(c for _, c in occurrence_counts) / len(occurrence_counts)
print(f"\nAverage occurrences per flagged page: {avg_occurrences:.1f}")

Total flagged pages: 302
Top 10 pages by occurrence count:
  Page 113: 2 occurrences
  Page 114: 2 occurrences
  Page 115: 2 occurrences
  Page 117: 2 occurrences
  Page 118: 2 occurrences
  Page 119: 2 occurrences
  Page 121: 2 occurrences
  Page 125: 2 occurrences
  Page 134: 2 occurrences
  Page 141: 2 occurrences

Average occurrences per flagged page: 1.6


In [6]:
def show_phrase_context(page_number: int, context_chars: int = 60) -> None:
    text = cleaned_pages[page_number - 1]["cleaned_text"]
    print(f"{'=' * 70}")
    print(f"PAGE {page_number} — all occurrences with surrounding context")
    print(f"{'=' * 70}")

    for match in re.finditer(re.escape(PHRASE), text):
        start = max(0, match.start() - context_chars)
        end = min(len(text), match.end() + context_chars)
        snippet = text[start:end].replace("\n", " \\n ")
        print(f"...{snippet}...")
        print("-" * 70)
    print()


# Show context on a couple of pages, ideally including the top offender
top_page = occurrence_counts[0][0] if occurrence_counts else None
if top_page:
    show_phrase_context(top_page)

# Also check page 13 specifically - it was supposed to be fully cleaned
show_phrase_context(13)

PAGE 113 — all occurrences with surrounding context
...ce, pedagogical competence, commitment, and integrity. More \n UNIVERSITY OF EDUCATION, LAHORE \n FALL \n UNIVERSITY OF EDUCATION, LAHORE \n FALL \n 107...
----------------------------------------------------------------------
...t, and integrity. More \n UNIVERSITY OF EDUCATION, LAHORE \n FALL \n UNIVERSITY OF EDUCATION, LAHORE \n FALL \n 107...
----------------------------------------------------------------------

PAGE 13 — all occurrences with surrounding context

